# Customer Protector Chatbot

In [43]:
# %pip install -q llama-index-core
# %pip install -q llama-index-readers-file
# %pip install -q llama-index-embeddings-huggingface
# %pip install -q llama-index-llms-openai-like
# %pip install -q pypdf
# %pip install -qqq llama-index-llms-groq
# %pip install -qqq llama-index-readers-file
# %pip install -qqq llama-index-embeddings-instructor
# %pip install python-dotenv
# %pip install --upgrade sympy
# %pip install jedi
# %pip install -Uqqq gradio==6.3.0

In [44]:
import os
import gradio as gr
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.base.llms.types import ChatMessage, MessageRole
from llama_index.core.chat_engine import ContextChatEngine
from pathlib import Path
from urllib.request import urlretrieve
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
# Load API keys from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv("HF_TOKEN")

# Initialize the Groq language model
llm = OpenAILike(
    model="openai/gpt-oss-120b",
    api_base="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    is_chat_model=True
)

# Initialize the local embedding model
embedding_dir = Path("embedding_model")
embedding_dir.mkdir(parents=True, exist_ok=True)

embeddings = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=str(embedding_dir),
)

# Initialize conversation memory
memory = ChatMemoryBuffer.from_defaults()
print("✅ Models and Memory successfully initialized!")

2026-10-29 22:48:41,261 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-10-29 22:48:42,800 - INFO - 2 prompts are loaded, with the keys: ['query', 'text']


✅ Models and Memory successfully initialized!


In [46]:
# Create the project data directory if it does not exist
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

# Prepare a plain-language summary of German consumer protection law
knowledge_text = """
GERMAN CONSUMER PROTECTION LAW (VERBRAUCHERSCHUTZRECHT) PRACTICAL SUMMARY

1. RIGHT OF WITHDRAWAL (WIDERRUFSRECHT - § 312g, § 355 BGB)
- Scope: Only for distance sales (online, telephone, catalogue) and off-premises contracts (doorstep sales).
- Physical Store Purchases (Ladengeschäft): NO statutory right to return or exchange items if they are not defective. Any return or refund in physical stores is purely voluntary customer service (Kulanz).
- Period: 14 calendar days starting the day after physical receipt of goods.
- Obligation: The consumer must declare withdrawal explicitly (email, cancellation form, portal). Sending goods back without notice is insufficient.
- Refunds: Full purchase price and standard outward delivery costs must be refunded within 14 days. Consumer pays return shipping only if notified before purchasing.
- Non-returnable Items (§ 312g Abs. 2 BGB): Custom/personalized items, sealed hygiene goods once unsealed, unsealed software/audio/video, and date-bound travel/event tickets.

2. STATUTORY DEFECT WARRANTY (GEWÄHRLEISTUNG - §§ 434, 437, 438, 477 BGB)
- Period: 2 years for newly purchased consumer goods from a business (B2C).
- Responsible Party: The SELLER (Verkäufer), not the manufacturer.
- Burden of Proof (§ 477 BGB): If a defect arises within the first 12 months, it is legally presumed to have existed at delivery. After 12 months, the consumer must prove the defect was present at delivery.
- Remedies (Nacherfüllung): Consumer can choose between repair or replacement. After two unsuccessful repair attempts or refusal by the seller, the consumer can cancel the contract (Rücktritt) for a full refund or demand a price reduction (Minderung).
- Private Sales: Private individuals (e.g., on Kleinanzeigen) can legally exclude Gewährleistung entirely, provided they do not fraudulently conceal known defects.

3. COMMERCIAL GUARANTEE (GARANTIE) VS. STATUTORY WARRANTY (GEWÄHRLEISTUNG)
- Gewährleistung is mandatory by federal statute (2 years against seller).
- Garantie is an optional contractual promise (usually by the manufacturer) and never weakens or replaces statutory Gewährleistung.

4. CONTRACTS & CANCELLATION BUTTON (§ 312k BGB)
- Online contracts (gyms, streaming, internet) must have a clearly visible cancellation button ("Kündigungsbutton").
- Automatic roll-overs for contracts signed after March 2022 allow monthly cancellation after the minimum term.
"""

# Save the consumer-law summary as a UTF-8 text file
text_path = data_dir / "consumer_rights_summary.txt"
text_path.write_text(knowledge_text, encoding="utf-8")

print(f"Consumer-law summary saved to: {text_path.resolve()}")

# Download the official English translation of the German Civil Code (BGB)
pdf_url = (
    "https://www.gesetze-im-internet.de/"
    "englisch_bgb/englisch_bgb.pdf"
)
pdf_path = data_dir / "german_civil_code_bgb.pdf"

try:
    if not pdf_path.exists():
        print("Downloading the BGB English translation PDF...")
        urlretrieve(pdf_url, pdf_path)

    print(f"PDF saved to: {pdf_path.resolve()}")

except Exception as error:
    print(f"PDF download failed: {error}")

print("Data preparation complete!")

Consumer-law summary saved to: C:\Users\Leyla\Desktop\WBS_DS_56\Machine_learning\GenerativeAI\data\consumer_rights_summary.txt
PDF saved to: C:\Users\Leyla\Desktop\WBS_DS_56\Machine_learning\GenerativeAI\data\german_civil_code_bgb.pdf
Data preparation complete!


In [47]:
# Load supported documents from the project data directory
data_dir = Path("data")

print(f"Loading documents from: {data_dir.resolve()}...")

documents = SimpleDirectoryReader(
    input_dir=str(data_dir),
    required_exts=[".txt", ".pdf"]
).load_data(show_progress=True)

print(f"Total document sections loaded: {len(documents)}")

# Split documents into overlapping text chunks
text_splitter = SentenceSplitter(
    chunk_size=600,
    chunk_overlap=120
)

# Build the vector index using the configured embedding model
print("\nGenerating embeddings and building vector index...")

vector_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    embed_model=embeddings,
    show_progress=True
)

print("Vector index created successfully.")

# # Persist the vector index for future use
# index_dir = Path("vector_index")
# index_dir.mkdir(parents=True, exist_ok=True)

# vector_index.storage_context.persist(
#     persist_dir=str(index_dir)
# )

# print(f"✅ Vector index saved to: {index_dir.resolve()}")


Loading documents from: C:\Users\Leyla\Desktop\WBS_DS_56\Machine_learning\GenerativeAI\data...


Loading files:   0%|          | 0/2 [00:00<?, ?it/s]

Total document sections loaded: 492

Generating embeddings and building vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/987 [00:00<?, ?it/s]

Vector index created successfully.


In [48]:
# Retrieve the five most relevant text chunks for each query
retriever = vector_index.as_retriever(similarity_top_k=5)

# Define the system instructions and response format
prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "You are a straightforward assistant specializing in "
            "German consumer rights law.\n\n"

            "Use the retrieved context as the primary source for "
            "your answers. Do not invent legal provisions, deadlines, "
            "exceptions, or consumer rights.\n"

            "If the retrieved context does not contain enough "
            "information to answer confidently, clearly state what "
            "is missing. Do not guess.\n"

            "Distinguish general legal information from conclusions "
            "that depend on the user's specific circumstances.\n\n"

            "Format EVERY response cleanly using ONLY these two sections:\n\n"

            "### ⚖️ Your Rights\n"
            "<Plain, bullet-point explanation of what the user can do,"
            "who pays, and what steps to take. Avoid unnecessary legal jargon.>\n\n"
            
            "### 📜 The Governing Law\n"
            "<List the specific sections of the German Civil Code (BGB) or"
            "regulations that give them these rights, with a one-sentence summary for each section.>\n\n"
            "Keep the answers short, direct, and actionable."
            "Do not add conversational fluff or introductory paragraphs."
        )
    )
]

# Initialize the context-aware chat engine
rag_bot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages
)

print("✅ Chatbot initialized with the two-section response format.")

✅ Chatbot initialized with the two-section response format.


## Interface

In [49]:
# Handle user messages and generate responses using the RAG chatbot
def custom_rag_bot_callback(message, history, top_k_value):
    # Update the number of retrieved text chunks
    rag_bot._retriever._similarity_top_k = int(top_k_value)

    # Reset the chat engine and restore the current conversation history
    rag_bot.reset()
    print(history)
    bot_history = [
        ChatMessage(role=m["role"], content=m["content"][0]["text"])
        for m in history
    ]
    response = rag_bot.chat(message, chat_history=bot_history)

    # Append the user's message and the assistant's response to the history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response.response})

    # Return the updated conversation history to the chatbot interface
    return history

In [50]:
# Export the chat history as a downloadable text file
def download_chat(history):
    # Check whether there are any messages to export
    if not history:
        return None
    
    # Create a list to store the formatted chat messages
    chat_text = []

    # Process each message in the conversation history
    for message in history:
        role = message.get("role", "Unknown").capitalize()
        content = message.get("content", "")

        # Handle messages stored as plain text
        if isinstance(content, str):
            text = content

        # Handle messages stored as a list of text content blocks
        elif isinstance(content, list):
            text = "\n".join(
                item.get("text", "")
                for item in content
                if isinstance(item, dict) and item.get("type") == "text"
            )
        
        # Convert other content types to text when possible
        else:
            text = str(content) if content else ""

        # Add non-empty messages to the export list
        if text.strip():
            chat_text.append(f"{role}:\n{text}\n")

    # Stop if no exportable messages were found
    if not chat_text:
        return None

    # Define the output file and save the conversation as UTF-8 text
    file_path = Path("chat_history.txt")
    file_path.write_text("\n".join(chat_text), encoding="utf-8")

    # Return the absolute file path for the Gradio download component
    return str(file_path.resolve())

In [56]:
custom_css = """
/* General background */
.gradio-container,
.gradio-container .main {
    background-color: #f7f5fb !important;
}

/* Header */
#app-header {
    text-align: center !important;
    color: #51416f !important;
}

/* Settings panel */
#settings-panel {
    background-color: #ece6f6 !important;
    border: 1px solid #d4c6e9 !important;
    border-radius: 14px !important;
    padding: 16px !important;
}

/* Chat panel */
#chat-panel {
    background-color: #ffffff !important;
    border: 2px solid #d4c6e9 !important;
    border-radius: 14px !important;
}

/* Chat message text - both user and assistant */
#chat-panel,
#chat-panel p,
#chat-panel span,
#chat-panel div {
    font-size: 14px !important;
    line-height: 1.5 !important;
}

/* Question input */
#question-input textarea {
    background-color: #ffffff !important;
    border: 1px solid #c2b0df !important;
    border-radius: 10px !important;
}

/* Buttons */
button {
    border-radius: 10px !important;
}

/* Primary buttons */
button.primary {
    background-color: #7961a8 !important;
    color: white !important;
}

/* Primary button hover */
button.primary:hover {
    background-color: #65508f !important;
}
"""

with gr.Blocks(title="Custom RAG Bot UI") as demo_custom:

    gr.Markdown(
        "# 💬 Customer Protector Assistant",
        elem_id="app-header"
    )

    with gr.Row(elem_id="main-row"):

        with gr.Column(
            scale=1,
            min_width=200,
            elem_id="settings-panel"
        ):
            gr.Markdown("### ⚙️ RAG Settings")

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=5,
                step=1,
                value=2,
                label="Top K",
                info="Number of relevant text chunks"
            )

            gr.Markdown(
                """
                **How it works**

                1. Choose an example or type your question.
                2. Retrieve relevant chunks.
                3. Generate an answer.
                """
            )

            clear_btn = gr.Button(
                "🗑️ Clear chat",
                variant="secondary"
            )

            download_btn = gr.Button(
                "💾 Download chat",
                variant="secondary"
            )

            download_file = gr.File(
                            label=None,
                            show_label=False,
                            height=60,
                            interactive=False,
                            visible=False
                        )
        with gr.Column(scale=4, min_width=0):

            chatbot = gr.Chatbot(
                # label="Customer Protector Assistant",
                height=350,
                value=[
                    {
                        "role": "assistant",
                        "content": (
                            "Hello! 👋 I am your RAG assistant. "
                            "Ask me anything about customer rights."
                        )
                    }
                ],
                elem_id="chat-panel",
                show_label=False
            )
            
            msg_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g. How many days do I have to return the product?",
                lines=1,
                max_lines=2,
                elem_id="question-input"
            )

            with gr.Row():
                submit_btn = gr.Button(
                    "Send message ➤",
                    variant="primary",
                    scale=4
                )

                clear_input_btn = gr.Button(
                    "Clear input",
                    scale=1
                )

            gr.Examples(
                examples=[
                    "They are refusing to accept my return",
                    "What if my product is defective?",
                    "Can I cancel my gym membership?"
                ],
                inputs=msg_input,
                label="💡 Example questions"
            )

    # Submit the message when the user presses Enter
    msg_input.submit(
        fn=custom_rag_bot_callback,
        inputs=[msg_input, chatbot, top_k_slider],
        outputs=chatbot
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=msg_input
    )

    # Submit the message when the user clicks the Send button
    submit_btn.click(
        fn=custom_rag_bot_callback,
        inputs=[msg_input, chatbot, top_k_slider],
        outputs=chatbot
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=msg_input
    )

    # Clear the conversation and restore the initial assistant message
    clear_btn.click(
        fn=lambda: [
            {
                "role": "assistant",
                "content": (
                    "Hello! 👋 I am your RAG assistant. "
                    "Ask me anything about customer rights."
                )
            }
        ],
        inputs=None,
        outputs=chatbot
    )

    # Clear only the question input field
    clear_input_btn.click(
        fn=lambda: "",
        inputs=None,
        outputs=msg_input
    )

    download_btn.click(
        fn=download_chat,
        inputs=chatbot,
        outputs=download_file
    ).then(
        fn=lambda: gr.update(visible=True),
        inputs=None,
        outputs=download_file
    )
    # download_btn.click(
    #     fn=download_chat,
    #     inputs=chatbot,
    #     outputs=gr.File(label="Download your chat")
    # )

demo_custom.launch(
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="slate"
    ),
    css=custom_css,
    share=True
)

2026-10-29 23:00:18,371 - INFO - HTTP Request: GET http://127.0.0.1:7879/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-10-29 23:00:18,392 - INFO - HTTP Request: HEAD http://127.0.0.1:7879/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7879


2026-10-29 23:00:18,647 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-10-29 23:00:19,166 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-10-29 23:00:19,271 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


[{'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! 👋 I am your RAG assistant. Ask me anything about customer rights.', 'type': 'text'}], 'options': None}]


2026-10-29 23:00:28,189 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
